# SentenceTransformers + CLIP

In [ ]:
# dense embedding
from sentence_transformers import SentenceTransformer
import numpy as np

text_model_name = "sentence-transformers/all-MiniLM-L6-v2"
text_model = SentenceTransformer(text_model_name)

sentences = [
    "The Metropolitan Museum of Art has a vast collection.",
    "A neural network learns from examples.",
    "The Metropolitan Museum of Art has a vast collection.",
]
emb = text_model.encode(sentences, convert_to_numpy=True)
print("Model:", text_model_name)
print("Embedding shape:", emb.shape)

# Cosine similarity
def cosine_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

print("cosine(sentence[0], sentence[2]):", cosine_sim(emb[0], emb[2]))
print("cosine(sentence[0], sentence[1]):", cosine_sim(emb[0], emb[1]))

Model: sentence-transformers/all-MiniLM-L6-v2
Embedding shape: (3, 384)
cosine(sentence[0], sentence[2]): 1.0
cosine(sentence[0], sentence[1]): 0.10113397985696793


In [ ]:
# clip

import torch
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

clip_name = "openai/clip-vit-base-patch32"
clip_model = CLIPModel.from_pretrained(clip_name)
clip_processor = CLIPProcessor.from_pretrained(clip_name)
clip_model.eval()

# Synthetic image test
image = Image.new("RGB", (224, 224), color=(220, 120, 40))
candidate_texts = [
    "a photograph of an orange sunset",
    "a close-up of a dog",
    "a solid orange color",
]

inputs = clip_processor(text=candidate_texts, images=image, return_tensors="pt", padding=True)
with torch.no_grad():
    out = clip_model(**inputs)

logits = out.logits_per_image[0]
probs = logits.softmax(dim=0)
print("CLIP:", clip_name)
for i, t in enumerate(candidate_texts):
    print(f"  P(text[{i}] | image) ≈ {probs[i].item():.4f}  — {t!r}")
best = int(probs.argmax())
print("Best matching caption index:", best, "→", candidate_texts[best])

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


CLIP: openai/clip-vit-base-patch32
  P(text[0] | image) ≈ 0.0090  — 'a photograph of an orange sunset'
  P(text[1] | image) ≈ 0.0003  — 'a close-up of a dog'
  P(text[2] | image) ≈ 0.9907  — 'a solid orange color'
Best matching caption index: 2 → a solid orange color
